## Load Dataframe

In [ ]:
!pip install -q jiwer sacrebleu openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 8.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_excel("/content/modernized_dataset.xlsx")

# LOWER
import re
import string

def normalize(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Create normalized columns
ORG_REF_COL = "sentence_human"
ORG_HYP_COL = "sentence_gpt4P4"

REF_COL = f"{ORG_REF_COL}_norm"
HYP_COL = f"{ORG_HYP_COL}_norm"

df[REF_COL] = df[ORG_REF_COL].astype(str).apply(normalize)
df[HYP_COL] = df[ORG_HYP_COL].astype(str).apply(normalize)

# Drop rows where either column is empty
df = df.dropna(subset=[REF_COL, HYP_COL])

# Convert to plain Python lists of strings
refs = df[REF_COL].astype(str).tolist()
hyps = df[HYP_COL].astype(str).tolist()

print(f"References : {len(refs)} sentences")
print(f"Hypotheses : {len(hyps)} sentences")

# Sanity check — first 3 pairs
for i in range(min(3, len(refs))):
    print(f"\n[{i+1}] REF: {refs[i]}")
    print(f"     HYP: {hyps[i]}")

References : 253 sentences
Hypotheses : 253 sentences

[1] REF: the state from storms secure by drowning proves now whilst despair doth doubtful fears appease he with the life the infirmitie removes thus is the physicke worse than the disease
     HYP: the state from storms secure by drowning proves now whilst despair doth doubtful fears appease he with the life the infirmity removes thus is the physic worse than the disease

[2] REF: be those the fancies of the inferior sort
     HYP: be those the fancies of the inferior sort

[3] REF: well since that to a private citizen you do ascribe the first most blessed estate
     HYP: well since that to a private citizen you do ascribe the first most blessed estate


## CER, WER sentence-level substitution, deletion, insertion

In [ ]:
#CER, WER sentence-level substitution, deletion, insertion

import jiwer
import pandas as pd

def get_alignment_str(ref: str, hyp: str) -> str:
    """Return a compact human-readable alignment string."""
    out = jiwer.process_words(ref, hyp)
    ref_words = ref.split()
    hyp_words = hyp.split()

    ref_idx = hyp_idx = 0
    parts = []

    for chunk in out.alignments[0]:
        op = chunk.type
        if op == "equal":
            for _ in range(chunk.ref_end_idx - chunk.ref_start_idx):
                parts.append(ref_words[ref_idx])
                ref_idx += 1; hyp_idx += 1
        elif op == "substitute":
            for _ in range(chunk.ref_end_idx - chunk.ref_start_idx):
                parts.append(f"[SUB: {ref_words[ref_idx]}→{hyp_words[hyp_idx]}]") #flags substituted word
                ref_idx += 1; hyp_idx += 1
        elif op == "delete":
            for _ in range(chunk.ref_end_idx - chunk.ref_start_idx):
                parts.append(f"[DEL:{ref_words[ref_idx]}]")                       #flags deleted word
                ref_idx += 1
        elif op == "insert":
            for _ in range(chunk.hyp_end_idx - chunk.hyp_start_idx):              #flags inserted word
                parts.append(f"[INS:{hyp_words[hyp_idx]}]")
                hyp_idx += 1

    return " ".join(parts)


def build_results_df(df: pd.DataFrame, ref_col: str, hyp_col: str) -> pd.DataFrame:

    # Original columns to carry over
    meta_cols = ["filename", "play_title", "speaker_id", "speaker_name",
                 "speaker_spblock", "gender", "sentence_number", "sample_index",
                 "description", "sentence_original", "sentence_gpt4P4"]

    records = []

    for _, row in df.iterrows():
        ref = str(row[ref_col])
        hyp = str(row[hyp_col])
        out = jiwer.process_words(ref, hyp)

        record = {col: row[col] for col in meta_cols if col in df.columns}  # original cols first

        record.update({
            "sentence_human":      ref,
            "sentence_matching":   get_alignment_str(ref, hyp),
            "wer":                 round(out.wer, 4),
            "cer":                 round(jiwer.cer(ref, hyp), 4),
            "substitutions":       out.substitutions,
            "deletions":           out.deletions,
            "insertions":          out.insertions,
            "ref_words":           len(ref.split()),
            "hyp_words":           len(hyp.split()),
        })
        records.append(record)

    results = pd.DataFrame(records)

    summary = {col: "" for col in meta_cols}  # empty for meta cols in summary row
    summary.update({
        "sentence_human":      "SUMMARY",
        "sentence_matching":   "",
        "wer":                 round(results["wer"].mean(), 4),
        "cer":                 round(results["cer"].mean(), 4),
        "substitutions":       results["substitutions"].sum(),
        "deletions":           results["deletions"].sum(),
        "insertions":          results["insertions"].sum(),
        "ref_words":           results["ref_words"].sum(),
        "hyp_words":           results["hyp_words"].sum(),
    })
    return pd.concat([results, pd.DataFrame([summary])], ignore_index=True)

# Build and export
results_df = build_results_df(df, REF_COL, HYP_COL)

## CER, WER, BLEU Scores

In [ ]:
#================
# CER, WER Scores
#================
import jiwer
import statistics

cer_scores = []
wer_scores = []

for ref, hyp in zip(refs, hyps):
    cer_scores.append(jiwer.cer(ref, hyp))
    wer_scores.append(jiwer.wer(ref, hyp))

def summarise(scores, label):
    print(f"{label}")
    print(f"  Mean   : {statistics.mean(scores):.4f}")
    print(f"  Median : {statistics.median(scores):.4f}")
    print(f"  Std dev: {statistics.stdev(scores):.4f}")
    print(f"  Min    : {min(scores):.4f}")
    print(f"  Max    : {max(scores):.4f}")

summarise(cer_scores, "CER")
summarise(wer_scores, "WER")

#================
# BLEU Score
#================

import sacrebleu

# Sentence-level BLEU
sbleu_scores = []
for ref, hyp in zip(refs, hyps):
    result = sacrebleu.sentence_bleu(hyp, [ref])
    sbleu_scores.append(result.score / 100.0)   # normalise to 0-1

# Corpus-level BLEU (more reliable — use this as the headline number)
corpus_result = sacrebleu.corpus_bleu(hyps, [refs])
corpus_bleu = corpus_result.score / 100.0

print(f"\nCorpus BLEU: {corpus_bleu:.4f}")

CER
  Mean   : 0.0218
  Median : 0.0000
  Std dev: 0.0733
  Min    : 0.0000
  Max    : 0.8571
WER
  Mean   : 0.0510
  Median : 0.0000
  Std dev: 0.1289
  Min    : 0.0000
  Max    : 1.0000

Corpus BLEU: 0.9656
